In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))
from regression_modelling.data_wrangling.dataset import build_model_table
from regression_modelling.models.model import fit_ols, coef_table, fit_summary
from regression_modelling.constants import PREDICTOR_COLS

In [2]:
# Build houston and chicago dataframes
hou = build_model_table("houston", refresh=True)
chi = build_model_table("chicago", refresh=True)
tables = {"houston": hou, "chicago": chi} 

Houston city boundary loaded: 1 row(s)
Houston state block groups loaded: 18,626


/home/eprashar_solutions_corelogic_com/crime-idx-2026/src/crime_blockgroup_mapping/boundaries.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bg['centroid'] = bg.geometry.centroid


Block groups within city: 1,629 / 18,626
Houston year_filter [2025-01-01, 2026-01-01): kept 240,696 of 240,696 rows (0 unparseable dates dropped)
Houston crime data: 237,011 rows with valid coords (of 240,696)
Matched to BG: 236,917 | Unmatched: 94
Mapped: 132,286 / 237,011 (55.8%)
Category counts:
crime_category
larceny     67789
vandal      20426
assault     13041
mvt         12355
burglary    11284
robbery      4973
rape         1921
murder        289
fire          208

Unmapped: 104,725 records across 38 codes
BG-level category aggregation: 2,079 BGs (1,600 within city, 479 outside)
Crime Totals:
assault_count      13032.0
murder_count         288.0
rape_count          1921.0
robbery_count       4971.0
burglary_count     11284.0
larceny_count      67775.0
mvt_count          12349.0
vandal_count       20421.0
violent_count      20212.0
property_count     91408.0
cl_total_count    111620.0
dtype: float64
Analysis set: 2,108 BGs
  Inside city:  1,629 (1,600 with data, 29 without)
  Ou

/home/eprashar_solutions_corelogic_com/crime-idx-2026/src/crime_blockgroup_mapping/boundaries.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bg['centroid'] = bg.geometry.centroid
/home/eprashar_solutions_corelogic_com/crime-idx-2026/src/crime_blockgroup_mapping/crime.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df[cfg.date_col], errors='coerce')


Chicago year_filter [2025-01-01, 2026-01-01): kept 237,207 of 237,207 rows (0 unparseable dates dropped)
Chicago crime data: 235,767 rows with valid coords (of 237,207)
Matched to BG: 235,690 | Unmatched: 77
Mapped: 129,757 / 235,767 (55.0%)
Category counts:
crime_category
larceny     58322
vandal      26091
mvt         17164
assault     13640
burglary     6167
robbery      5780
rape         1792
murder        431
fire          370

Unmapped: 106,010 records across 15 codes
BG-level category aggregation: 2,256 BGs (2,164 within city, 92 outside)
Crime Totals:
assault_count      13636.0
murder_count         429.0
rape_count          1792.0
robbery_count       5775.0
burglary_count      6167.0
larceny_count      58295.0
mvt_count          17163.0
vandal_count       26081.0
violent_count      21632.0
property_count     81625.0
cl_total_count    103257.0
dtype: float64
Analysis set: 2,256 BGs
  Inside city:  2,164 (2,164 with data, 0 without)
  Outside city: 92 (all with crime data)
Model 

#### Separate regressions for Houston and Chicago

In [3]:
# # Run regression on Houston crime
predictors = PREDICTOR_COLS
result, robust, houston_reg = fit_ols(
    df=hou, 
    target="cl_total_logcount"
    )
print(result.summary())
print("="*80)
print(fit_summary(result).to_string(), "\n")
tab = coef_table(result, robust, predictors)
print(tab.to_string())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.259
Model:                            OLS   Adj. R-squared:                  0.253
Method:                 Least Squares   F-statistic:                     47.03
Date:                Tue, 18 Aug 2026   Prob (F-statistic):           2.70e-96
Time:                        23:07:55   Log-Likelihood:                -2332.4
No. Observations:                1629   AIC:                             4691.
Df Residuals:                    1616   BIC:                             4761.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const           

In [4]:
# Run regression on Chicago crime
predictors = PREDICTOR_COLS
result, robust, chicago_reg = fit_ols(
    df=chi, 
    target="cl_total_logcount"
    )
print(result.summary())
print("="*80)
print(fit_summary(result).to_string(), "\n")
tab = coef_table(result, robust, predictors)
print(tab.to_string())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.302
Model:                            OLS   Adj. R-squared:                  0.298
Method:                 Least Squares   F-statistic:                     77.61
Date:                Tue, 18 Aug 2026   Prob (F-statistic):          2.90e-158
Time:                        23:08:11   Log-Likelihood:                -2309.6
No. Observations:                2164   AIC:                             4645.
Df Residuals:                    2151   BIC:                             4719.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const           